# 06. 면접 피드백 생성 v2 (영상·텍스트 입력)

05(영상)·04(텍스트)와 같은 흐름에 검색·프롬프트만 바꾼 버전입니다.

| | v1 (04·05) | v2 (이 노트북) |
|---|---|---|
| 참고자료 DB | 영상 전사 33개 (모의면접 전사 포함) | **강의 영상 74개만** (`rag_v2.py`) |
| 검색 | 임베딩 | **키워드 + 임베딩 순위 결합** |
| 프롬프트 | 발표자료 slide19 | slide19 규칙 + 판단 쏠림·형식 보정 (`prompts_v2.py`) |
| 피드백 예시 | `data/feedback_data` 무작위 5개 | 같음 (시드 고정 가능) |

평가 결과: `eval_v2.py` → `eval_results_v2/`

In [ ]:
%pip install google-genai chromadb sentence-transformers scikit-learn python-dotenv --quiet

In [ ]:
import os, re, time, random
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

BASE = Path(r'C:\Users\82105\OneDrive\바탕 화면\interview-coach')
PROJ = BASE.parent / '프로젝트3(면접)'
FB_DIR = BASE / 'data' / 'feedback_data'   # Few-Shot용

load_dotenv(PROJ / '.env')
client = genai.Client(api_key=os.environ.get('GOOGLE_API_KEY', ''))

# 발표자료 기준 모델은 gemini-3.1-pro-preview (무료 등급은 한도 0 → 결제 필요)
MODEL = 'gemini-3.5-flash'
# 생각(thinking) 토큰도 출력 한도에 포함되므로 넉넉히 둔다 (1,200이면 답이 잘리거나 비어 나올 수 있음)
MAX_TOKENS = 4096
print(f'설정 완료 | 모델: {MODEL}')

## 피드백 예시 (Few-Shot) — 모의면접 답변 ↔ 코치 피드백 31쌍

In [ ]:
pairs = []
for grp, ivd, fbd in (('1~5', 'Interview_text', 'Feedback_text'), ('11~16', 'interview_text', 'feedback_text')):
    for iv in sorted((FB_DIR / grp / ivd).glob('*.txt')):
        fb = FB_DIR / grp / fbd / iv.name
        if fb.exists():
            pairs.append({'interview': iv.read_text(encoding='utf-8').strip(),
                          'feedback': fb.read_text(encoding='utf-8').strip()})
text_dir = FB_DIR / '6~10' / 'text'
for iv in sorted(text_dir.glob('*인터뷰*.txt')):
    n = re.match(r'(\d+)', iv.name)
    if not n:
        continue
    fb = '\n\n'.join(f.read_text(encoding='utf-8').strip() for f in sorted(text_dir.glob(f'{n.group(1)}_[0-9]*.txt')))
    pairs.append({'interview': iv.read_text(encoding='utf-8').strip(), 'feedback': fb})
pairs = [p for p in pairs if p['interview'] and p['feedback']]
print(f'Few-Shot 예시: {len(pairs)}쌍')

## 참고자료 검색 (RAG v2)

In [ ]:
from rag_v2 import build_db, HybridRetriever
from prompts_v2 import system_prompt

build_db()                 # 최초 1회 구축, 이후엔 있는 DB 사용
rag = HybridRetriever()

In [ ]:
def build_contents(n_shots, seed):
    shots = random.Random(seed).sample(pairs, min(n_shots, len(pairs)))
    contents = []
    for s in shots:   # Q./A. 형식 (발표자료 slide12 방식)
        contents.append(types.Content(role='user', parts=[types.Part.from_text(text=f'Q.\n{s["interview"]}')]))
        contents.append(types.Content(role='model', parts=[types.Part.from_text(text=f'A.\n{s["feedback"]}')]))
    return contents


def rag_block(hits):
    return '--- 참고 자료 (면접왕 이형 강의) ---\n' + '\n\n'.join(f'[참고{i+1}] {h["text"]}' for i, h in enumerate(hits))


def generate(contents, video=False):
    resp = client.models.generate_content(model=MODEL, contents=contents, config=types.GenerateContentConfig(
        system_instruction=system_prompt(video=video), temperature=0.4, max_output_tokens=MAX_TOKENS))
    # 예시를 Q./A. 형식으로 넣어서 답변 앞에 'A.'를 붙이는 경우가 있어 떼어 낸다
    return re.sub(r'^\s*A\.\s*', '', resp.text or '')


def get_feedback(answer, n_shots=5, n_rag=4, seed=None):
    '''텍스트 입력: 답변 → 하이브리드 검색 → 예시 + 근거 + 규칙 → 피드백'''
    hits = rag.search(answer[:300], n_rag)
    contents = build_contents(n_shots, seed)
    contents.append(types.Content(role='user', parts=[types.Part.from_text(text=f'Q.\n{answer}\n\n{rag_block(hits)}')]))
    return {'feedback': generate(contents), 'rag_sources': [h['source'] for h in hits]}


def get_video_feedback(video_path, n_shots=5, n_rag=4, seed=None):
    '''영상 입력: 업로드 → 핵심 키워드 추출 → 하이브리드 검색 → 영상 + 예시 + 근거 + 규칙 → [MM:SS] 피드백'''
    video = client.files.upload(file=str(video_path), config={'mime_type': 'video/mp4'})
    while video.state.name == 'PROCESSING':
        time.sleep(5)
        video = client.files.get(name=video.name)
    if video.state.name == 'FAILED':
        raise RuntimeError(f'영상 처리 실패: {video.state}')

    pre = client.models.generate_content(model=MODEL, contents=[video, '이 영상에서 지원자가 말하는 핵심 키워드를 5개만 추출해줘. 단어만.'],
                                         config=types.GenerateContentConfig(max_output_tokens=MAX_TOKENS))
    keywords = pre.text or ''
    hits = rag.search(keywords, n_rag)
    contents = build_contents(n_shots, seed)
    contents.append(types.Content(role='user', parts=[
        types.Part.from_uri(file_uri=video.uri, mime_type='video/mp4'),
        types.Part.from_text(text=f'Q.\n위 면접 영상을 분석해줘.\n\n{rag_block(hits)}')]))
    feedback = generate(contents, video=True)
    client.files.delete(name=video.name)   # 면접 영상은 개인 정보라 사용 후 서버에서 지운다
    return {'feedback': feedback, 'keywords': keywords, 'rag_sources': [h['source'] for h in hits]}

print('준비 완료')

## 텍스트 입력

In [ ]:
ANSWER = '''안녕하세요. 저는 성실하고 책임감 있는 사람입니다.
학교에서 열심히 공부했고, 팀 프로젝트도 많이 해봤습니다.
이 회사에 오고 싶어서 지원했습니다. 잘 부탁드립니다.'''

result = get_feedback(ANSWER, seed=0)
print('참고 강의:', result['rag_sources'])
print(result['feedback'])

## 영상 입력

In [ ]:
VIDEO_PATH = r'C:\Users\82105\OneDrive\바탕 화면\drive_unzip\최종테스트_영상\21.mp4'

result = get_video_feedback(VIDEO_PATH, seed=0)
print('키워드:', result['keywords'])
print('참고 강의:', result['rag_sources'])
print(result['feedback'])